In [1]:
import torch
import pandas as pd
import numpy as np
from torchmetrics import Accuracy
from algorithms.pso import PSO
from algorithms.ga import GA
from algorithms.cmaes import CMAES
from algorithms.de import DE
import matplotlib.pyplot as plt

from algorithms.minimize import minimize

In [2]:
np.random.seed(42)

In [3]:
device = "cpu" 

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda" 

device

'mps'

In [4]:
data = pd.read_csv("Data/winequality-red.csv", sep=";")

X_np = data[data.columns[:-1]].values
y_np = data[data.columns[-1]].values
y_np = y_np.astype("float") + np.random.lognormal(size=y_np.shape[0])
# y_np = np.random.lognormal(size=y_np.shape[0])

In [5]:
X = torch.tensor(X_np, dtype=torch.float32).to(device)
y = torch.tensor(y_np, dtype=torch.float32).to(device) 

In [6]:
loss = torch.nn.MSELoss()

In [7]:
def MLP(theta, x, h_units):
        """tiny MLP"""
        
        _, n_features = x.shape
        
        w1, b1, w2, b2 = torch.split(theta, [n_features*h_units, h_units, h_units, 1])
        h = torch.tanh(x @ w1.view(n_features,h_units) + b1)
        out = h @ w2.view(h_units, 1) + b2
        return out

def fitness_function(x, h_units=32):

    def obj(pop):
        fitnesses = []
        for theta in pop:
            
            logits = MLP(theta, x, h_units)
            
            fitness = loss(logits, y)
            fitnesses.append(fitness)
        return torch.stack(fitnesses)
    return obj

In [8]:
n_classes  = len(y.unique())
n_features = X.shape[-1]
h_units    = 128

In [9]:
n_weights = n_features*h_units + h_units + h_units + 1

In [10]:
obj_function = fitness_function(X, h_units=h_units)

In [11]:
lower_bound = [-10]*n_weights
upper_bound = [10]*n_weights

In [12]:
pop_size = 30
max_evals = pop_size*100

# Adam

In [13]:
n_epochs = max_evals

lb = torch.tensor(lower_bound, device=device)
ub = torch.tensor(upper_bound, device=device)

mean = 0.5 * (lb + ub)          # centre of the box
std  = 0.5 * (ub - lb) / 3.0    # 3-σ rule  ⇒  99.7 % inside bounds
weights = mean + std * torch.randn(1, n_weights, device=device)
weights = torch.max(torch.min(weights, ub), lb)

weights = torch.nn.Parameter(weights.squeeze(0))
optimizer = torch.optim.Adam([weights], lr=0.001) 

In [14]:
for epoch in range(n_epochs):
    optimizer.zero_grad(set_to_none=True)
    
    logits = MLP(weights, X, h_units=h_units)
    l = loss(logits, y)
    print(f"Epoch {epoch+1:4d} | Loss = {l.item():.4f}")
    l.backward()
    optimizer.step()

/Users/tangherloni/anaconda3/envs/AI/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([1599])) that is different to the input size (torch.Size([1599, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch    1 | Loss = 242.0526
Epoch    2 | Loss = 238.6525
Epoch    3 | Loss = 235.2901
Epoch    4 | Loss = 231.9706
Epoch    5 | Loss = 228.7000
Epoch    6 | Loss = 225.4836
Epoch    7 | Loss = 222.3246
Epoch    8 | Loss = 219.2249
Epoch    9 | Loss = 216.1867
Epoch   10 | Loss = 213.2134
Epoch   11 | Loss = 210.3097
Epoch   12 | Loss = 207.4799
Epoch   13 | Loss = 204.7263
Epoch   14 | Loss = 202.0476
Epoch   15 | Loss = 199.4391
Epoch   16 | Loss = 196.8936
Epoch   17 | Loss = 194.4021
Epoch   18 | Loss = 191.9552
Epoch   19 | Loss = 189.5433
Epoch   20 | Loss = 187.1576
Epoch   21 | Loss = 184.7903
Epoch   22 | Loss = 182.4353
Epoch   23 | Loss = 180.0915
Epoch   24 | Loss = 177.7661
Epoch   25 | Loss = 175.4706
Epoch   26 | Loss = 173.2113
Epoch   27 | Loss = 170.9898
Epoch   28 | Loss = 168.8074
Epoch   29 | Loss = 166.6676
Epoch   30 | Loss = 164.5744
Epoch   31 | Loss = 162.5286
Epoch   32 | Loss = 160.5269
Epoch   33 | Loss = 158.5640
Epoch   34 | Loss = 156.6384
Epoch   35 | L

# PSO

In [15]:
algorithm = PSO(obj_function,
                dim=n_weights,
                pop_size=pop_size,
                lower_bound=lower_bound,
                upper_bound=upper_bound,
                initialisation="gaussian",
                init_v_max=1.,
                init_v_min=-1.,
                device=device)

minimize(algorithm, max_evals=max_evals, verbose=True)

Generation    1 | Loss = 88.7156, best_f = 88.7156
Generation    2 | Loss = 71.6846, best_f = 71.6846
Generation    3 | Loss = 56.1676, best_f = 56.1676
Generation    4 | Loss = 68.1450, best_f = 56.1676
Generation    5 | Loss = 25.2188, best_f = 25.2188
Generation    6 | Loss = 42.9856, best_f = 25.2188
Generation    7 | Loss = 35.3682, best_f = 25.2188
Generation    8 | Loss = 38.3252, best_f = 25.2188
Generation    9 | Loss = 29.3819, best_f = 25.2188
Generation   10 | Loss = 29.3068, best_f = 25.2188
Generation   11 | Loss = 26.5446, best_f = 25.2188
Generation   12 | Loss = 27.4969, best_f = 25.2188
Generation   13 | Loss = 25.0965, best_f = 25.0965
Generation   14 | Loss = 22.8485, best_f = 22.8485
Generation   15 | Loss = 22.4719, best_f = 22.4719
Generation   16 | Loss = 20.6181, best_f = 20.6181
Generation   17 | Loss = 22.0452, best_f = 20.6181
Generation   18 | Loss = 20.5623, best_f = 20.5623
Generation   19 | Loss = 20.9970, best_f = 20.5623
Generation   20 | Loss = 16.448

In [16]:
from fstpso import FuzzyPSO

In [17]:
def fitness_function_fst_pso(x, h_units=32):

    def obj(ind):
        theta = torch.tensor(ind, dtype=torch.float32).to(device)
        logits = MLP(theta, x, h_units)    
        fitness = loss(logits, y)
        return fitness.cpu().item()
    return obj

In [18]:
obj_function_fst_pso = fitness_function_fst_pso(X, h_units=h_units)

In [19]:
FP = FuzzyPSO()
FP.set_search_space(list(zip(lower_bound, upper_bound)))
FP.set_swarm_size(pop_size)
FP.set_fitness(obj_function_fst_pso)
FP.max_evaluations = max_evals

sigma  = 0.5 * (upper_bound[0] - lower_bound[0]) / 3.0


result = FP.solve_with_fstpso(creation_method={'name':"normal", "sigma":sigma},)

Fuzzy Self-Tuning PSO - v1.8.1
 * Max distance: 816.088231
 * Search space boundaries set to: [(-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-10, 10), (-1

# GAs

In [20]:
algorithm = GA(obj_function,
               dim=n_weights,
               pop_size=pop_size,
               lower_bound=lower_bound,
               upper_bound=upper_bound,
               initialisation="uniform",
               mutation="gaussian",
               crossover="blend",
               device=device)

minimize(algorithm, max_evals=max_evals, verbose=True)


Generation    1 | Loss = 317.2861, best_f = 317.2861
Generation    2 | Loss = 311.5148, best_f = 311.5148
Generation    3 | Loss = 271.4948, best_f = 271.4948
Generation    4 | Loss = 256.7312, best_f = 256.7312
Generation    5 | Loss = 232.9447, best_f = 232.9447
Generation    6 | Loss = 216.4086, best_f = 216.4086
Generation    7 | Loss = 194.8425, best_f = 194.8425
Generation    8 | Loss = 180.4086, best_f = 180.4086
Generation    9 | Loss = 163.8969, best_f = 163.8969
Generation   10 | Loss = 149.6863, best_f = 149.6863
Generation   11 | Loss = 146.2000, best_f = 146.2000
Generation   12 | Loss = 138.4667, best_f = 138.4667
Generation   13 | Loss = 135.2541, best_f = 135.2541
Generation   14 | Loss = 129.3742, best_f = 129.3742
Generation   15 | Loss = 122.9610, best_f = 122.9610
Generation   16 | Loss = 120.1856, best_f = 120.1856
Generation   17 | Loss = 112.2236, best_f = 112.2236
Generation   18 | Loss = 111.7523, best_f = 111.7523
Generation   19 | Loss = 99.4635, best_f = 99.

# DE

In [21]:
algorithm = DE(obj_function,
               dim=n_weights,
               pop_size=pop_size,
               lower_bound=lower_bound,
               upper_bound=upper_bound,
               initialisation="gaussian",
               device=device
               )

minimize(algorithm, max_evals=max_evals, verbose=True)

Generation    1 | Loss = 132.2897, best_f = 132.2897
Generation    2 | Loss = 120.9603, best_f = 120.9603
Generation    3 | Loss = 111.8706, best_f = 111.8706
Generation    4 | Loss = 109.4748, best_f = 109.4748
Generation    5 | Loss = 109.3268, best_f = 109.3268
Generation    6 | Loss = 104.8694, best_f = 104.8694
Generation    7 | Loss = 102.2861, best_f = 102.2861
Generation    8 | Loss = 101.4822, best_f = 101.4822
Generation    9 | Loss = 100.7971, best_f = 100.7971
Generation   10 | Loss = 99.8273, best_f = 99.8273
Generation   11 | Loss = 99.6568, best_f = 99.6568
Generation   12 | Loss = 98.8200, best_f = 98.8200
Generation   13 | Loss = 98.8200, best_f = 98.8200
Generation   14 | Loss = 97.7107, best_f = 97.7107
Generation   15 | Loss = 96.8205, best_f = 96.8205
Generation   16 | Loss = 95.6550, best_f = 95.6550
Generation   17 | Loss = 95.2570, best_f = 95.2570
Generation   18 | Loss = 95.2339, best_f = 95.2339
Generation   19 | Loss = 93.7318, best_f = 93.7318
Generation   